# JetClass Dataset: Merge

This notebook loads the 10 JetClass jet types from individual ROOT files and merges them into a single parquet file with only the constituent four-momenta and a jet type label.

It expects one ROOT file per jet type (named `*_100.root`) from the JetClass test set, placed in the `jetclass_test_files/` subfolder.

In [ ]:
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
import vector
import pathlib

vector.register_awkward()

%matplotlib inline
%config InlineBackend.figure_format='retina'

## Configuration and file discovery

In [ ]:
dataset_path = pathlib.Path(".")

# Maximum number of jets to keep per jet type (set to None for no limit)
N_JETS_PER_TYPE = 1_000

In [ ]:
# List the root files in jetclass_test_files
jet_class_path = dataset_path / "jetclass_test_files"
root_files = sorted(jet_class_path.glob("*.root"))
print(f"Found {len(root_files)} files:")
for f in root_files:
    print(f"  {f.name}")

## Load and merge all jet types

Each ROOT file corresponds to one jet type and contains a TTree called `tree`.
We load the constituent four-momenta (`part_px/py/pz/energy`) using `uproot`, build `Momentum4D` arrays, and concatenate everything into a single awkward array. A `jet_type_id` array (integer label) and a `jet_type_names` list allow selecting individual jet types later.

In [ ]:
import uproot

all_constituents = []
all_jet_type_ids = []
jet_type_names = []

for type_id, rf in enumerate(root_files):
    # Derive a human-readable jet type name from the filename
    # e.g. "ZJetsToNuNu_100.root" -> "ZJetsToNuNu"
    name = rf.stem.rsplit("_", 1)[0]
    jet_type_names.append(name)

    tree = uproot.open(rf)["tree"]
    entry_stop = N_JETS_PER_TYPE if N_JETS_PER_TYPE is not None else tree.num_entries
    branches = tree.arrays(
        ["part_px", "part_py", "part_pz", "part_energy"],
        entry_stop=entry_stop,
    )
    n_jets = len(branches)
    print(f"[{type_id}] {name}: {n_jets} jets (of {tree.num_entries} available)")

    constituents = ak.zip(
        {
            "px": branches["part_px"],
            "py": branches["part_py"],
            "pz": branches["part_pz"],
            "E": branches["part_energy"],
        },
        with_name="Momentum4D",
    )
    all_constituents.append(constituents)
    all_jet_type_ids.append(np.full(n_jets, type_id, dtype=np.int32))

# Concatenate all jet types into single arrays
jet_constituents = ak.concatenate(all_constituents, axis=0)
jet_type_id = np.concatenate(all_jet_type_ids)

print(f"\nTotal jets: {len(jet_constituents)}")
print("\nJet type mapping:")
for i, name in enumerate(jet_type_names):
    count = np.sum(jet_type_id == i)
    print(f"  {i}: {name} ({count} jets)")

## Quick sanity check

Verify the merged array by plotting the number of constituents per jet for
each jet type.

In [ ]:
n_constituents = ak.num(jet_constituents)

fig, ax = plt.subplots(figsize=(8, 4))
hist_kwargs = dict(bins=np.arange(0, 130, 2), histtype="step", density=True)

for i, name in enumerate(jet_type_names):
    mask = jet_type_id == i
    ax.hist(np.array(n_constituents[mask]), **hist_kwargs, label=name)

ax.set_xlabel("Number of constituents")
ax.set_ylabel("Density")
ax.legend(frameon=False, ncol=2, fontsize=8)
ax.set_title("Constituent multiplicity by jet type")
plt.tight_layout()
plt.show()

## Save merged dataset to parquet

Store the merged dataset as a single parquet file containing only the
constituent four-momenta (`part_px`, `part_py`, `part_pz`, `part_energy`)
and the `jet_type_id` / `jet_type_name` labels.

In [ ]:
# Build a single awkward array with all fields
merged = ak.zip(
    {
        "part_px": jet_constituents.px,
        "part_py": jet_constituents.py,
        "part_pz": jet_constituents.pz,
        "part_energy": jet_constituents.E,
        "jet_type_id": jet_type_id,
        "jet_type_name": np.array([jet_type_names[i] for i in jet_type_id]),
    },
    depth_limit=1,
)

output_path = dataset_path / f"MiniJetClass_merged_{N_JETS_PER_TYPE:_}_per_type.parquet"
ak.to_parquet(merged, output_path)

file_size_mb = output_path.stat().st_size / 1024**2
print(f"Saved merged dataset to {output_path}")
print(f"File size: {file_size_mb:.1f} MB")